# 阶段 6：验证 MLP 推理结果上板可用

本 notebook 做三件事：

1. 确认 overlay 是阶段 6 的硬件（`VERSION = 0x00010006`）；
2. 让视频流跑起来，观察 `MLP_OUTPUT` 是否随场景变化；
3. 顺手确认视频直通没被破坏。

**操作**：运行下面的监控单元后，依次做：遮住镜头 → 对着窗/灯 → 恢复正常并移动物体，
观察打印出来的 `mlp` 是否分别趋向 0 / 1 / 2。

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

# 找工程根目录（含 lib/video_setup.py 的那一层）。
# 为什么不写死路径：板子上工程可能叫 ai_video，也可能叫 ai_video_2024.2；
# 而且 Jupyter 的 cwd 有时是 notebook 所在目录、有时是启动服务时的目录 ——
# 实测踩过：cwd = /home/xilinx/jupyter_notebooks 时，`from lib import ...` 直接
# ModuleNotFoundError: No module named 'lib'，而且重跑整个 notebook 会一直挂在这格。
def _find_root():
    cands = [os.path.abspath(os.getcwd())]
    for base in ("/home/xilinx/jupyter_notebooks", os.path.dirname(os.path.abspath(os.getcwd()))):
        if os.path.isdir(base):
            try:
                cands += [os.path.join(base, n) for n in sorted(os.listdir(base))]
            except OSError:
                pass
    for start in cands:
        d = start
        for _ in range(6):
            if os.path.isfile(os.path.join(d, "lib", "video_setup.py")):
                return d
            nd = os.path.dirname(d)
            if nd == d:
                break
            d = nd
    return None


ROOT = _find_root()
if ROOT is None:
    raise RuntimeError("找不到工程根目录（含 lib/video_setup.py 的那一层）。"
                       "先 cd 到 ai_video_2024.2 目录再启动 Jupyter，或把本 notebook 放进工程里运行。")
sys.path.insert(0, ROOT)
print("工程根目录:", ROOT)

from lib import video_setup as vs
from lib import video_path as vp
from lib.ai_video_regs import AiVideoRegs, STATUS_DONE

# 演示/验证必须和采集用**同一档锁定曝光**，否则特征会整体偏移：
# 采集时是 736 行 + 增益 0x20（见 docs/04_demo_box.md 的记录）。
# 不锁的话 AEC/AGC 会自己改曝光/增益 -> 噪声变 -> f_grad 被抬高十几个点，
# 三类中心就对不上（实测：自动曝光下常规档读 (109,57,36)，训练中心是 (96.5,68.6,29.9)）。
# 重新采集过就改成新的工作点。
# 画面太暗/全黑时先改下面这个开关：False = 退回自动曝光（画面会亮起来，
# 但特征会偏移、三类中心对不上，只适合「只想看一眼画面」的场合）。
USE_MANUAL_EXPOSURE = True
DEMO_EXPOSURE, DEMO_GAIN = 736, 0x20      # 采集时锁定的工作点，重采过就改这里
ol, iic, demosaic, vdma0, vdma1, mode = vs.bring_up(
    ROOT, manual_exposure=(dict(exposure=DEMO_EXPOSURE, gain=DEMO_GAIN)
                           if USE_MANUAL_EXPOSURE else None))
vs.stop_all_channels(vdma0, vdma1)
regs = AiVideoRegs(ol.axis_video_proc_0)
print("VERSION = 0x%08X  (阶段 %d)" % (regs.version(), regs.version_stage()))
assert regs.version_stage() >= 6, "当前 overlay 不是阶段 6 的硬件"

# 只启动采集，保证视频流经过 axis_video_proc（统计与推理才会跑）
vp.start_capture(vdma0, mode)
regs.wait_new_frame()
print("视频流已启动，FRAME_CNT =", regs.frame_cnt())

## 监控 MLP_OUTPUT

下面这个单元默认跑 30 秒（约 60 次/秒读一次）。期间请按提示切换场景。

In [ ]:
def monitor(seconds=30.0, poll=0.2):
    t0 = time.time()
    hist = {}
    while time.time() - t0 < seconds:
        mlp = regs.mlp_output()
        st = regs.status()
        f1, f2, f3 = regs.mlp_features()
        hist[mlp] = hist.get(mlp, 0) + 1
        print("t=%5.1fs  MLP_OUTPUT=%d  done=%d  f=(%3d,%6d,%3d)"
              % (time.time() - t0, mlp, 1 if (st & STATUS_DONE) else 0, f1, f2, f3))
        time.sleep(poll)
    print("统计:", hist)
    return hist

hist = monitor(seconds=30.0)

## 确认视频直通没被破坏

抓一帧显示出来。注意：这一版 IP 还没有改像素（改像素是阶段 7），
所以画面应该和阶段 3 时一模一样。

In [ ]:
raw = vp.capture_frame(vdma0)
rgb = vp.frame_to_rgb(raw)
print("frame shape:", raw.shape, " max:", int(raw.max()))
plt.figure(figsize=(10, 6))
plt.imshow(rgb)
plt.title("stage6: MLP integrated, video still byte-identical passthrough")
plt.axis("off")
plt.show()

## 结论

* 三个场景下 `MLP_OUTPUT` 明显以不同值为主 → 阶段 6 完成。
* 如果三个场景都读出同一个值：先看 `f=(...)` 是否随场景变化。
  * 不变化 → 是采集/统计的问题（回阶段 3）。
  * 变化但输出不变 → 是握手/权重的问题（检查 `csim` 是否通过、`ap_rst` 是否接对）。

In [ ]:
import gc

def safe_shutdown(ol=None, vdma0=None, vdma1=None, vs=None):
    """收尾：停 VDMA 通道 + 尽量释放 overlay。任何一步失败都不影响后面。

    为什么不再写 demosaic.stop / iic.stop：
    v_demosaic_0 和 axi_iic_0 只是 AXI-Lite 寄存器空间，**根本没有 .stop() 方法**。
    上一版写成 _try(demosaic.stop, ...) 会报 AttributeError —— 因为属性取值
    （demosaic.stop）发生在调用 _try 之前，异常在“传参那一刻”就抛出来了，
    _try 里面的 try 根本来不及接住。真正需要停的只有 VDMA 通道。
    """
    def _try(fn, name):
        try:
            fn()
            print('[ok]   %s' % name)
        except Exception as e:
            print('[warn] %s: %s' % (name, e))

    # 1) 停 VDMA 通道（唯一必须做的一步）
    if vs is not None and vdma0 is not None:
        _try(lambda: vs.stop_all_channels(vdma0, vdma1), 'stop_all_channels')
    elif vdma0 is not None:
        _try(vdma0.readchannel.stop, 'vdma0.readchannel.stop')

    # 2) 释放 overlay（PYNQ 各版本接口名不一样，失败也只是打个 warn）
    if ol is not None:
        _try(lambda: ol.free(), 'overlay.free')

    gc.collect()
    print('✅ 硬件资源已释放')

safe_shutdown(ol=ol, vdma0=vdma0, vdma1=vdma1, vs=vs)